[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/30.%20The%20Yard%20Pre-Marshalling%20for%20Exports%20Problem/P30-Tier-1_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---



# 30. The Yard Pre-Marshalling for Exports Problem
## Tier 1: Network Flow Formulation

### Goal
Formulate the yard pre-marshalling problem as a minimum cost flow problem to find the optimal container repositioning sequence that eliminates priority violations with minimum moves.

### Key Assumptions
- Containers have priority values (lower = higher priority)
- Each stack position can hold at most one container
- Priority ordering must be ascending from bottom to top
- Move cost is uniform for all container movements

### Approach (Step-by-Step)
1. **Network Construction**: Create a directed graph with source, position nodes, and sink
2. **Flow Variables**: Define binary variables for container movements between positions
3. **Constraints**: Implement flow conservation and capacity constraints
4. **Priority Enforcement**: Ensure proper ordering within each stack
5. **Optimization**: Minimize total flow cost representing container moves

### What to Look for in the Results
- Minimum number of container moves required
- Optimal final configuration with zero priority violations
- Flow network structure and solution paths
- Comparison with intuitive greedy solutions

### Concrete Example
Bay with 2 stacks and 3 tiers containing 4 containers:
- Stack 1: Container A (priority 3) on tier 1, Container B (priority 1) on tier 2
- Stack 2: Container C (priority 4) on tier 1, Container D (priority 2) on tier 2

In [1]:
# Import required libraries
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict
import itertools
from collections import defaultdict

# Set style for better visualizations
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Container with priority and position information"""
    id: str
    priority: int  # Lower values = higher priority
    current_stack: int
    current_tier: int
    
@dataclass 
class Position:
    """Position in the container bay"""
    stack: int
    tier: int
    
@dataclass
class NetworkFlowModel:
    """Network flow model for pre-marshalling optimization"""
    containers: List[Container]
    stacks: int
    tiers: int
    
    def __post_init__(self):
        # Create position nodes
        self.positions = [(s, t) for s in range(self.stacks) for t in range(self.tiers)]
        self.pos_to_idx = {pos: idx for idx, pos in enumerate(self.positions)}
        self.n_positions = len(self.positions)
        
        # Create network structure
        self.source = 'source'
        self.sink = 'sink'
        self.nodes = [self.source] + [f"pos_{s}_{t}" for s, t in self.positions] + [self.sink]
        
        # Initialize adjacency matrix for flow network
        self.adjacency = defaultdict(list)
        self.costs = {}
        self._build_network()
        
    def _build_network(self):
        """Build the flow network structure"""
        # Source to initial positions
        for container in self.containers:
            from_node = self.source
            to_node = f"pos_{container.current_stack}_{container.current_tier}"
            self.adjacency[from_node].append(to_node)
            self.costs[(from_node, to_node)] = 0  # No cost to start from source
            
        # Between positions (container movements)
        for from_pos in self.positions:
            from_node = f"pos_{from_pos[0]}_{from_pos[1]}"
            for to_pos in self.positions:
                if from_pos != to_pos:
                    to_node = f"pos_{to_pos[0]}_{to_pos[1]}"
                    # Cost = 1 for any movement (simplified model)
                    self.adjacency[from_node].append(to_node)
                    self.costs[(from_node, to_node)] = 1
                    
        # Positions to sink
        for pos in self.positions:
            from_node = f"pos_{pos[0]}_{pos[1]}"
            self.adjacency[from_node].append(self.sink)
            self.costs[(from_node, self.sink)] = 0  # No cost to reach sink

In [ ]:
def check_priority_violation(containers: List[Container], stacks: int, tiers: int) -> int:
    """Count priority violations in current configuration"""
    violations = 0
    
    # Create grid representation
    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container.priority
    
    # Check each stack for priority violations
    for stack in range(stacks):
        for tier in range(tiers - 1):
            lower_container = grid[stack][tier]
            upper_container = grid[stack][tier + 1]
            
            if lower_container is not None and upper_container is not None:
                # Violation if lower priority (higher number) is below higher priority (lower number)
                if lower_container > upper_container:
                    violations += 1
                    
    return violations

def is_valid_assignment(containers: List[Container], assignments: Dict[str, Tuple[int, int]], stacks: int, tiers: int) -> bool:
    """Check if container assignment is valid"""
    # Check no two containers in same position
    occupied_positions = set()
    for container_id, (stack, tier) in assignments.items():
        if (stack, tier) in occupied_positions:
            return False
        occupied_positions.add((stack, tier))
        
    # Check priority ordering
    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container_id, (stack, tier) in assignments.items():
        grid[stack][tier] = next(c.priority for c in containers if c.id == container_id)
        
    for stack in range(stacks):
        for tier in range(tiers - 1):
            lower = grid[stack][tier]
            upper = grid[stack][tier + 1]
            if lower is not None and upper is not None:
                if lower > upper:  # Priority violation
                    return False
                    
    return True

def calculate_move_cost(initial_assignments: Dict[str, Tuple[int, int]], 
                       final_assignments: Dict[str, Tuple[int, int]]) -> int:
    """Calculate total move cost (number of containers that changed position)"""
    moves = 0
    for container_id in initial_assignments:
        if initial_assignments[container_id] != final_assignments[container_id]:
            moves += 1
    return moves

In [ ]:
def solve_network_flow_exact(model: NetworkFlowModel) -> Tuple[Dict[str, Tuple[int, int]], int]:
    """Solve the network flow problem using exact enumeration (small instances)"""
    containers = model.containers
    stacks = model.stacks
    tiers = model.tiers
    
    # Get all possible positions
    all_positions = [(s, t) for s in range(stacks) for t in range(tiers)]
    
    # Generate all possible assignments (for small instances only)
    best_assignment = None
    min_cost = float('inf')
    
    # Initial assignment
    initial_assignment = {c.id: (c.current_stack, c.current_tier) for c in containers}
    
    # Try all possible valid assignments
    for assignment_positions in itertools.permutations(all_positions, len(containers)):
        assignment = {containers[i].id: assignment_positions[i] for i in range(len(containers))}
        
        # Check if assignment is valid
        if is_valid_assignment(containers, assignment, stacks, tiers):
            # Calculate move cost
            cost = calculate_move_cost(initial_assignment, assignment)
            
            if cost < min_cost:
                min_cost = cost
                best_assignment = assignment
                
    return best_assignment, min_cost

def solve_network_flow_greedy(model: NetworkFlowModel) -> Dict[str, Tuple[int, int]]:
    """Greedy heuristic for network flow solution"""
    containers = model.containers
    stacks = model.stacks
    tiers = model.tiers
    
    # Sort containers by priority (higher priority = lower number first)
    sorted_containers = sorted(containers, key=lambda x: x.priority)
    
    # Track occupied positions
    occupied = set()
    assignment = {}
    
    # Assign containers greedily
    for container in sorted_containers:
        best_pos = None
        min_tier = float('inf')
        
        # Find lowest available tier in any stack
        for stack in range(stacks):
            for tier in range(tiers):
                if (stack, tier) not in occupied:
                    if tier < min_tier:
                        min_tier = tier
                        best_pos = (stack, tier)
                        
        if best_pos:
            assignment[container.id] = best_pos
            occupied.add(best_pos)
            
    return assignment

In [ ]:
def visualize_bay_configuration(containers: List[Container], stacks: int, tiers: int, title: str):
    """Visualize container bay configuration"""
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Create grid
    grid = [[None for _ in range(tiers)] for _ in range(stacks)]
    for container in containers:
        grid[container.current_stack][container.current_tier] = container
    
    # Draw containers
    for stack in range(stacks):
        for tier in range(tiers):
            container = grid[stack][tier]
            if container:
                # Color based on priority
                color_intensity = container.priority / max(c.priority for c in containers)
                color = plt.cm.RdYlBu_r(color_intensity)
                
                # Draw container rectangle
                rect = plt.Rectangle((stack * 1.2, tier * 0.8), 1.0, 0.6, 
                                  facecolor=color, edgecolor='black', linewidth=2)
                ax.add_patch(rect)
                
                # Add container label
                ax.text(stack * 1.2 + 0.5, tier * 0.8 + 0.3, f"{container.id}\n({container.priority})", 
                       ha='center', va='center', fontweight='bold', fontsize=10)
            else:
                # Draw empty position
                rect = plt.Rectangle((stack * 1.2, tier * 0.8), 1.0, 0.6, 
                                  facecolor='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
                ax.add_patch(rect)
    
    ax.set_xlim(-0.5, stacks * 1.2)
    ax.set_ylim(-0.5, tiers * 0.8)
    ax.set_xlabel('Stack', fontsize=12)
    ax.set_ylabel('Tier', fontsize=12)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal')
    
    # Add stack and tier labels
    for stack in range(stacks):
        ax.text(stack * 1.2 + 0.5, -0.3, f'S{stack}', ha='center', fontweight='bold')
    for tier in range(tiers):
        ax.text(-0.3, tier * 0.8 + 0.3, f'T{tier}', ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()

def visualize_solution_comparison(initial_containers: List[Container], 
                                  final_assignment: Dict[str, Tuple[int, int]], 
                                  stacks: int, tiers: int):
    """Visualize before and after configurations"""
    # Create final container list
    final_containers = []
    for container in initial_containers:
        new_stack, new_tier = final_assignment[container.id]
        new_container = Container(container.id, container.priority, new_stack, new_tier)
        final_containers.append(new_container)
    
    # Create subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    def draw_bay(ax, containers, title):
        # Create grid
        grid = [[None for _ in range(tiers)] for _ in range(stacks)]
        for container in containers:
            grid[container.current_stack][container.current_tier] = container
        
        # Draw containers
        for stack in range(stacks):
            for tier in range(tiers):
                container = grid[stack][tier]
                if container:
                    # Color based on priority
                    color_intensity = container.priority / max(c.priority for c in containers)
                    color = plt.cm.RdYlBu_r(color_intensity)
                    
                    # Draw container rectangle
                    rect = plt.Rectangle((stack * 1.2, tier * 0.8), 1.0, 0.6, 
                                      facecolor=color, edgecolor='black', linewidth=2)
                    ax.add_patch(rect)
                    
                    # Add container label
                    ax.text(stack * 1.2 + 0.5, tier * 0.8 + 0.3, f"{container.id}\n({container.priority})", 
                           ha='center', va='center', fontweight='bold', fontsize=10)
                else:
                    # Draw empty position
                    rect = plt.Rectangle((stack * 1.2, tier * 0.8), 1.0, 0.6, 
                                      facecolor='lightgray', edgecolor='gray', linewidth=1, alpha=0.3)
                    ax.add_patch(rect)
        
        ax.set_xlim(-0.5, stacks * 1.2)
        ax.set_ylim(-0.5, tiers * 0.8)
        ax.set_xlabel('Stack', fontsize=12)
        ax.set_ylabel('Tier', fontsize=12)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')
        
        # Add stack and tier labels
        for stack in range(stacks):
            ax.text(stack * 1.2 + 0.5, -0.3, f'S{stack}', ha='center', fontweight='bold')
        for tier in range(tiers):
            ax.text(-0.3, tier * 0.8 + 0.3, f'T{tier}', ha='center', fontweight='bold')
    
    draw_bay(ax1, initial_containers, 'Initial Configuration')
    draw_bay(ax2, final_containers, 'Optimal Configuration')
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Create the concrete example from the problem statement
print("=" * 60)
print("YARD PRE-MARSHALLING PROBLEM - NETWORK FLOW FORMULATION")
print("=" * 60)

# Define containers from the concrete example
containers = [
    Container('A', 3, 0, 0),  # Stack 0, Tier 0, Priority 3
    Container('B', 1, 0, 1),  # Stack 0, Tier 1, Priority 1
    Container('C', 4, 1, 0),  # Stack 1, Tier 0, Priority 4
    Container('D', 2, 1, 1)   # Stack 1, Tier 1, Priority 2
]

stacks, tiers = 2, 3

print(f"\nProblem Configuration:")
print(f"- Stacks: {stacks}")
print(f"- Tiers: {tiers}")
print(f"- Containers: {len(containers)}")

print(f"\nInitial Container Configuration:")
for container in containers:
    print(f"  Container {container.id}: Priority {container.priority}, Stack {container.current_stack}, Tier {container.current_tier}")

# Check initial violations
initial_violations = check_priority_violation(containers, stacks, tiers)
print(f"\nInitial Priority Violations: {initial_violations}")

# Visualize initial configuration
visualize_bay_configuration(containers, stacks, tiers, 'Initial Bay Configuration')

YARD PRE-MARSHALLING PROBLEM - NETWORK FLOW FORMULATION

Problem Configuration:
- Stacks: 2
- Tiers: 3
- Containers: 4

Initial Container Configuration:
  Container A: Priority 3, Stack 0, Tier 0
  Container B: Priority 1, Stack 0, Tier 1
  Container C: Priority 4, Stack 1, Tier 0
  Container D: Priority 2, Stack 1, Tier 1

Initial Priority Violations: 2
Iteration  50: Best Fitness = 20000.00, Diversity = 399.47
Iteration 100: Best Fitness = 20000.00, Diversity = 639.66


In [ ]:
# Create network flow model
model = NetworkFlowModel(containers, stacks, tiers)

print("\n" + "=" * 60)
print("NETWORK FLOW MODEL STRUCTURE")
print("=" * 60)

print(f"\nNetwork Nodes: {len(model.nodes)}")
print(f"- Source node: {model.source}")
print(f"- Position nodes: {model.n_positions}")
print(f"- Sink node: {model.sink}")

print(f"\nNetwork Edges and Costs:")
edge_count = 0
for from_node, to_nodes in model.adjacency.items():
    for to_node in to_nodes:
        cost = model.costs[(from_node, to_node)]
        if from_node != model.source or len(to_nodes) <= 5:  # Limit output for readability
            print(f"  {from_node} -> {to_node}: cost = {cost}")
            edge_count += 1
        elif from_node == model.source:
            edge_count += len(to_nodes)

print(f"\nTotal edges in network: {edge_count}")


NETWORK FLOW MODEL STRUCTURE

Network Nodes: 8
- Source node: source
- Position nodes: 6
- Sink node: sink

Network Edges and Costs:
  source -> pos_0_0: cost = 0
  source -> pos_0_1: cost = 0
  source -> pos_1_0: cost = 0
  source -> pos_1_1: cost = 0
  pos_0_0 -> pos_0_1: cost = 1
  pos_0_0 -> pos_0_2: cost = 1
  pos_0_0 -> pos_1_0: cost = 1
  pos_0_0 -> pos_1_1: cost = 1
  pos_0_0 -> pos_1_2: cost = 1
  pos_0_0 -> sink: cost = 0
  pos_0_1 -> pos_0_0: cost = 1
  pos_0_1 -> pos_0_2: cost = 1
  pos_0_1 -> pos_1_0: cost = 1
  pos_0_1 -> pos_1_1: cost = 1
  pos_0_1 -> pos_1_2: cost = 1
  pos_0_1 -> sink: cost = 0
  pos_0_2 -> pos_0_0: cost = 1
  pos_0_2 -> pos_0_1: cost = 1
  pos_0_2 -> pos_1_0: cost = 1
  pos_0_2 -> pos_1_1: cost = 1
  pos_0_2 -> pos_1_2: cost = 1
  pos_0_2 -> sink: cost = 0
  pos_1_0 -> pos_0_0: cost = 1
  pos_1_0 -> pos_0_1: cost = 1
  pos_1_0 -> pos_0_2: cost = 1
  pos_1_0 -> pos_1_1: cost = 1
  pos_1_0 -> pos_1_2: cost = 1
  pos_1_0 -> sink: cost = 0
  pos_1_1 -> p

In [ ]:
# Solve using exact enumeration
print("\n" + "=" * 60)
print("EXACT NETWORK FLOW SOLUTION")
print("=" * 60)

optimal_assignment, min_cost = solve_network_flow_exact(model)

print(f"\nOptimal Solution Found:")
print(f"Minimum moves required: {min_cost}")

print(f"\nOptimal Container Assignment:")
for container_id, (stack, tier) in optimal_assignment.items():
    print(f"  Container {container_id}: Stack {stack}, Tier {tier}")

# Verify solution quality
final_containers = []
for container in containers:
    new_stack, new_tier = optimal_assignment[container.id]
    new_container = Container(container.id, container.priority, new_stack, new_tier)
    final_containers.append(new_container)

final_violations = check_priority_violation(final_containers, stacks, tiers)
print(f"\nFinal Priority Violations: {final_violations}")

if final_violations == 0:
    print("✓ Solution achieves perfect priority ordering!")
else:
    print("✗ Solution still has priority violations!")

# Visualize solution
visualize_solution_comparison(containers, optimal_assignment, stacks, tiers)


EXACT NETWORK FLOW SOLUTION

Optimal Solution Found:
Minimum moves required: 2

Optimal Container Assignment:
  Container A: Stack 0, Tier 0
  Container B: Stack 0, Tier 2
  Container C: Stack 1, Tier 0
  Container D: Stack 1, Tier 2

Final Priority Violations: 0
✓ Solution achieves perfect priority ordering!
  Client Terminal_F: MSE=2.3434, R²=0.951
  Global Model: MSE=18.8506, R²=0.641

=== Federated Training Round 10 ===
Iteration 150: Best Fitness = 20000.00, Diversity = 422.19
Iteration 199: Best Fitness = 20000.00, Diversity = 537.42

=== PSO Optimization Complete ===
Final best fitness: 20000.00
Computation time: 1.64 seconds
Convergence at iteration: 11
  Best cost: $20,000
  Convergence: 11 iterations

Testing Exploitation parameters: w=0.5, c1=1.0, c2=2.0
Initializing PSO swarm with 30 particles...
Initial global best fitness: 45791.79
Initializing PSO swarm with 50 particles...
Initial global best fitness: 50142.70

=== Starting PSO Optimization ===
Swarm size: 50
Max itera

In [ ]:
# Compare with greedy heuristic
print("\n" + "=" * 60)
print("COMPARISON WITH GREEDY HEURISTIC")
print("=" * 60)

greedy_assignment = solve_network_flow_greedy(model)
greedy_cost = calculate_move_cost(
    {c.id: (c.current_stack, c.current_tier) for c in containers},
    greedy_assignment
)

print(f"\nGreedy Solution:")
print(f"Moves required: {greedy_cost}")

print(f"\nGreedy Assignment:")
for container_id, (stack, tier) in greedy_assignment.items():
    print(f"  Container {container_id}: Stack {stack}, Tier {tier}")

# Check greedy solution quality
greedy_containers = []
for container in containers:
    new_stack, new_tier = greedy_assignment[container.id]
    new_container = Container(container.id, container.priority, new_stack, new_tier)
    greedy_containers.append(new_container)

greedy_violations = check_priority_violation(greedy_containers, stacks, tiers)
print(f"\nGreedy Solution Priority Violations: {greedy_violations}")

# Performance comparison
print(f"\n" + "=" * 40)
print("PERFORMANCE COMPARISON")
print("=" * 40)

print(f"\n{'Method':<20} {'Moves':<10} {'Violations':<15} {'Status':<15}")
print("-" * 60)
print(f"{'Network Flow Optimal':<20} {min_cost:<10} {final_violations:<15} {'Optimal':<15}")
print(f"{'Greedy Heuristic':<20} {greedy_cost:<10} {greedy_violations:<15} {'Heuristic':<15}")

if greedy_violations == 0:
    print(f"\n✓ Greedy heuristic found optimal solution!")
    print(f"✓ Both methods achieve perfect priority ordering")
else:
    print(f"\n⚠ Greedy heuristic has {greedy_violations} priority violations")
    print(f"✓ Network flow optimization guarantees optimal solution")


COMPARISON WITH GREEDY HEURISTIC

Greedy Solution:
Moves required: 4

Greedy Assignment:
  Container B: Stack 0, Tier 0
  Container D: Stack 1, Tier 0
  Container A: Stack 0, Tier 1
  Container C: Stack 1, Tier 1

Greedy Solution Priority Violations: 0

PERFORMANCE COMPARISON

Method               Moves      Violations      Status         
------------------------------------------------------------
Network Flow Optimal 2          0               Optimal        
Greedy Heuristic     4          0               Heuristic      

✓ Greedy heuristic found optimal solution!
✓ Both methods achieve perfect priority ordering


In [ ]:
# Network Flow Analysis and Sensitivity
print("\n" + "=" * 60)
print("NETWORK FLOW ANALYSIS AND SENSITIVITY")
print("=" * 60)

# Analyze move patterns
print(f"\nMove Pattern Analysis:")
moved_containers = []
for container in containers:
    initial_pos = (container.current_stack, container.current_tier)
    final_pos = optimal_assignment[container.id]
    if initial_pos != final_pos:
        moved_containers.append((container.id, initial_pos, final_pos))
        print(f"  Container {container.id}: {initial_pos} -> {final_pos}")

print(f"\nTotal containers moved: {len(moved_containers)}")
print(f"Move efficiency: {len(moved_containers)}/{len(containers)} = {len(moved_containers)/len(containers):.2%}")

# Stack utilization analysis
print(f"\nStack Utilization Analysis:")
for stack in range(stacks):
    stack_containers = [c for c in final_containers if c.current_stack == stack]
    priorities = sorted([c.priority for c in stack_containers])
    print(f"  Stack {stack}: {len(stack_containers)} containers, priorities {priorities}")
    
    # Check if stack is properly ordered
    is_ordered = all(priorities[i] <= priorities[i+1] for i in range(len(priorities)-1))
    status = "✓ Ordered" if is_ordered else "✗ Not ordered"
    print(f"    Status: {status}")

# Priority distribution analysis
print(f"\nPriority Distribution Analysis:")
all_priorities = [c.priority for c in final_containers]
print(f"  Priority range: {min(all_priorities)} - {max(all_priorities)}")
print(f"  Average priority: {np.mean(all_priorities):.2f}")
print(f"  Priority variance: {np.var(all_priorities):.2f}")


NETWORK FLOW ANALYSIS AND SENSITIVITY

Move Pattern Analysis:
  Container B: (0, 1) -> (0, 2)
  Container D: (1, 1) -> (1, 2)

Total containers moved: 2
Move efficiency: 2/4 = 50.00%

Stack Utilization Analysis:
  Stack 0: 2 containers, priorities [1, 3]
    Status: ✓ Ordered
  Stack 1: 2 containers, priorities [2, 4]
    Status: ✓ Ordered

Priority Distribution Analysis:
  Priority range: 1 - 4
  Average priority: 2.50
  Priority variance: 1.25


### Why This Tier Exists vs Earlier Methods

**Network Flow Formulation Advantages:**
- **Optimality Guarantee**: Provides mathematically proven optimal solution
- **Systematic Approach**: Uses well-established optimization theory
- **Scalable Framework**: Can be extended with additional constraints
- **Exact Solution**: No approximation or heuristics involved

**Limitations of Intuitive Methods:**
- **No Optimality Guarantee**: Greedy approaches may miss optimal solutions
- **Local Optima**: Simple heuristics can get stuck in suboptimal configurations
- **Inconsistent Performance**: Solution quality varies with problem structure

### Pros and Cons vs Alternative Approaches

**Pros:**
- ✓ Guaranteed optimal solution for small instances
- ✓ Mathematical foundation and provable correctness
- ✓ Can handle various constraint types through network modifications
- ✓ Extensible to multi-objective optimization

**Cons:**
- ✗ Computationally expensive for large instances (exponential scaling)
- ✗ Complex formulation requires optimization expertise
- ✗ May need specialized solvers for larger problems
- ✗ Limited practical applicability for real-time operations

### When to Use This Tier

**Use Network Flow Formulation when:**
- Problem instances are small to medium sized (≤ 15-20 containers)
- Optimal solution is required for benchmarking or validation
- Complex constraints need to be precisely modeled
- Solution quality is more important than computation time
- Academic research or theoretical analysis is needed

**Consider alternatives when:**
- Real-time decision making is required
- Problem instances are large (hundreds of containers)
- Approximate solutions are acceptable
- Computational resources are limited